# VECTOR HOÁ TF-IDF VÀ CÁC MÔ HÌNH BASELINE (V3)

In [1]:
import os
import re
import sys
import joblib
import pandas as pd
import numpy as np
from scipy.special import expit

In [2]:
ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
PROC_DIR = os.path.join(ROOT, 'data', 'processed')
MODEL_DIR = os.path.join(ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)
if not os.environ.get('TEMP'):
    os.environ['TEMP'] = os.path.join(ROOT, '.tmp_run')
os.makedirs(os.environ['TEMP'], exist_ok=True)
os.environ['TMP'] = os.environ['TEMP']

In [3]:
#Nap helper dung chung tu src/ de dong nhat chia Dev 3 phan va suy dien Multi-Label Aspect
from src.labels import ASPECT_GROUPS
from src.modeling import split_dev_three_way, multilabel_to_primary_aspect

In [4]:
#Doc 3 tap du lieu sach tu buoc 01
df_train = pd.read_csv(os.path.join(PROC_DIR, 'train_clean.csv'))
df_dev = pd.read_csv(os.path.join(PROC_DIR, 'dev_clean.csv'))
df_test = pd.read_csv(os.path.join(PROC_DIR, 'test_clean.csv'))

In [5]:
df_train.shape, df_dev.shape, df_test.shape

((7776, 16), (1110, 16), (2224, 16))

In [6]:
asp_cols = ['asp_0', 'asp_1', 'asp_2', 'asp_3', 'asp_4']
df_train[['sentence_seg', 'sentiment', 'aspect_group'] + asp_cols].head(3)

,sentence_seg,sentiment,aspect_group,asp_0,asp_1,asp_2,asp_3,asp_4
0,Mới mua máy này Tại thegioididong thốt_nốt cảm...,2,2,1,1,1,1,1
1,Pin kém còn lại miễn chê mua 8 / 3 / 2019 tình...,1,2,0,0,1,0,1
2,Sao lúc gọi điện_thoại màn_hình bị chấm nhỏ nh...,0,0,1,0,0,0,0


## CHIA TẬP DEV 3 PHẦN ĐỘC LẬP (EYEBALL, CALIB, BLACKBOX)

In [7]:
#Chia dev thanh Eyeball (250), Calib (300), Blackbox (560); tuyet doi khong dung Blackbox khi tune CV
df_eyeball, df_calib, df_blackbox = split_dev_three_way(df_dev, random_state=42)
df_eyeball.shape, df_calib.shape, df_blackbox.shape

((250, 16), (300, 16), (560, 16))

In [8]:
pd.DataFrame({
    'eyeball': df_eyeball['sentiment'].value_counts(),
    'calib': df_calib['sentiment'].value_counts(),
    'blackbox': df_blackbox['sentiment'].value_counts(),
}).sort_index()

,eyeball,calib,blackbox
sentiment,,,
0,78,96,176
1,48,58,110
2,124,146,274


In [9]:
pd.DataFrame({
    'eyeball': df_eyeball['aspect_group'].value_counts(),
    'calib': df_calib['aspect_group'].value_counts(),
    'blackbox': df_blackbox['aspect_group'].value_counts(),
}).sort_index()

,eyeball,calib,blackbox
aspect_group,,,
0,104,125,234
1,32,38,72
2,53,63,118
3,33,40,73
4,28,34,63


## TRỌNG SỐ LỚP LÀM MỊN VÀ ĐẶC TRƯNG TF-IDF

In [10]:
#Tinh can bac hai nghich dao tan suat lop cho sentiment de chong overfit lop thieu so
s_cnt = df_train['sentiment'].value_counts().to_dict()
smooth_w_sent = {c: float(np.sqrt(len(df_train) / (3.0 * count))) for c, count in s_cnt.items()}
smooth_w_sent

{2: 0.833052161400748, 0: 1.0147893268214596, 1: 1.3041424008039932}

In [11]:
#Tinh trong so lam min cho 5 nhom khia canh
a_cnt = df_train['aspect_group'].value_counts().to_dict()
smooth_w_asp = {c: float(np.sqrt(len(df_train) / (5.0 * count))) for c, count in a_cnt.items()}
smooth_w_asp

{0: 0.6981194240089781,
 2: 0.9738032054408269,
 3: 1.191208672815045,
 1: 1.2890067732709791,
 4: 1.3051419079092288}

In [12]:
X_tr_text = df_train['sentence_seg'].fillna('')
X_eye_text = df_eyeball['sentence_seg'].fillna('')
X_cal_text = df_calib['sentence_seg'].fillna('')
X_blk_text = df_blackbox['sentence_seg'].fillna('')
X_te_text = df_test['sentence_seg'].fillna('')

In [13]:
y_tr_sent, y_eye_sent, y_blk_sent, y_te_sent = (
    df_train['sentiment'].values, df_eyeball['sentiment'].values,
    df_blackbox['sentiment'].values, df_test['sentiment'].values
)
y_tr_asp, y_eye_asp, y_blk_asp, y_te_asp = (
    df_train['aspect_group'].values, df_eyeball['aspect_group'].values,
    df_blackbox['aspect_group'].values, df_test['aspect_group'].values
)

In [14]:
Y_tr_asp = df_train[asp_cols].values
Y_eye_asp = df_eyeball[asp_cols].values
Y_blk_asp = df_blackbox[asp_cols].values
Y_te_asp = df_test[asp_cols].values
Y_tr_asp.shape, Y_blk_asp.shape, Y_te_asp.shape

((7776, 5), (560, 5), (2224, 5))

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import f1_score, accuracy_score, classification_report

In [16]:
#Giu token 1 ky tu ('1'..'5', 'o', 'a', 'y') va PII tags ([PHONE], [ORDER_ID], [EMAIL], [URL], [PERSON])
tfidf_params = dict(
    token_pattern=r'(?u)\b\w+\b|\[\w+\]',
    ngram_range=(1, 2),
    max_features=15000,
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    lowercase=True,
)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [17]:
#Chi fit TF-IDF doc lap tren df_train['sentence_seg'] de phuc vu SHAP/tuong thich, khong cham Dev/Test
vec = TfidfVectorizer(**tfidf_params)
X_tr_tfidf = vec.fit_transform(X_tr_text)
vocab_check = {t: (t in vec.vocabulary_) for t in ['1', '5', 'ở', 'ạ', 'ý', '[phone]', '[url]', '[person]']}
X_tr_tfidf.shape, vocab_check

((7776, 15000),
 {'1': True,
  '5': True,
  'ở': True,
  'ạ': True,
  'ý': True,
  '[phone]': True,
  '[url]': True,
  '[person]': True})

## TINH CHỈNH MÔ HÌNH CẢM XÚC BẰNG PIPELINE + STRATIFIEDKFOLD(5)

In [18]:
from sklearn.naive_bayes import ComplementNB
#1. ComplementNB trong Pipeline + StratifiedKFold(5) thuan tren df_train
pipe_nb = Pipeline([('tfidf', TfidfVectorizer(**tfidf_params)), ('clf', ComplementNB())])
nb_grid = {'clf__alpha': [0.05, 0.1, 0.25, 0.5, 1.0], 'clf__norm': [True, False]}
nb_search = GridSearchCV(pipe_nb, nb_grid, cv=skf, scoring='f1_macro', n_jobs=-1, refit=True)
nb_search.fit(X_tr_text, y_tr_sent)
nb_sent_pipe = nb_search.best_estimator_
print('NB best params:', nb_search.best_params_)

NB best params: {'clf__alpha': 0.1, 'clf__norm': True}


In [19]:
f1_train = f1_score(y_tr_sent, nb_sent_pipe.predict(X_tr_text), average='macro')
f1_cv = float(nb_search.best_score_)
f1_eyeball = f1_score(y_eye_sent, nb_sent_pipe.predict(X_eye_text), average='macro')
f1_blackbox = f1_score(y_blk_sent, nb_sent_pipe.predict(X_blk_text), average='macro')
print(f'NB Sent -> Train F1: {f1_train:.4f} | CV F1: {f1_cv:.4f} | Eyeball F1: {f1_eyeball:.4f} | Blackbox F1: {f1_blackbox:.4f}')
nb_sent_scores = {'model': 'ComplementNB', 'pipe': nb_sent_pipe, 'train_f1': f1_train, 'cv_f1': f1_cv, 'eyeball_f1': f1_eyeball, 'blackbox_f1': f1_blackbox}

NB Sent -> Train F1: 0.8880 | CV F1: 0.6997 | Eyeball F1: 0.7090 | Blackbox F1: 0.7324


In [20]:
from sklearn.linear_model import LogisticRegression
#2. LogisticRegression trong Pipeline + StratifiedKFold(5) thuan tren df_train
pipe_lr = Pipeline([('tfidf', TfidfVectorizer(**tfidf_params)), ('clf', LogisticRegression(max_iter=1000, random_state=42))])
lr_grid = {'clf__C': [0.2, 0.5, 1.0, 2.0, 5.0], 'clf__class_weight': [None, 'balanced', smooth_w_sent]}
lr_search = GridSearchCV(pipe_lr, lr_grid, cv=skf, scoring='f1_macro', n_jobs=-1, refit=True)
lr_search.fit(X_tr_text, y_tr_sent)
lr_sent_pipe = lr_search.best_estimator_
print('LR best params:', lr_search.best_params_)

LR best params: {'clf__C': 1.0, 'clf__class_weight': 'balanced'}


In [21]:
f1_train = f1_score(y_tr_sent, lr_sent_pipe.predict(X_tr_text), average='macro')
f1_cv = float(lr_search.best_score_)
f1_eyeball = f1_score(y_eye_sent, lr_sent_pipe.predict(X_eye_text), average='macro')
f1_blackbox = f1_score(y_blk_sent, lr_sent_pipe.predict(X_blk_text), average='macro')
print(f'LR Sent -> Train F1: {f1_train:.4f} | CV F1: {f1_cv:.4f} | Eyeball F1: {f1_eyeball:.4f} | Blackbox F1: {f1_blackbox:.4f}')
lr_sent_scores = {'model': 'LogisticRegression', 'pipe': lr_sent_pipe, 'train_f1': f1_train, 'cv_f1': f1_cv, 'eyeball_f1': f1_eyeball, 'blackbox_f1': f1_blackbox}

LR Sent -> Train F1: 0.9111 | CV F1: 0.7310 | Eyeball F1: 0.7363 | Blackbox F1: 0.7621


In [22]:
from sklearn.svm import LinearSVC
#3. LinearSVC trong Pipeline + StratifiedKFold(5) thuan tren df_train
pipe_svc = Pipeline([('tfidf', TfidfVectorizer(**tfidf_params)), ('clf', LinearSVC(max_iter=3000, random_state=42))])
svc_grid = {'clf__C': [0.05, 0.1, 0.2, 0.5, 1.0], 'clf__class_weight': [None, 'balanced', smooth_w_sent]}
svc_search = GridSearchCV(pipe_svc, svc_grid, cv=skf, scoring='f1_macro', n_jobs=-1, refit=True)
svc_search.fit(X_tr_text, y_tr_sent)
svc_sent_pipe = svc_search.best_estimator_
print('SVC best params:', svc_search.best_params_)

SVC best params: {'clf__C': 0.2, 'clf__class_weight': 'balanced'}


In [23]:
f1_train = f1_score(y_tr_sent, svc_sent_pipe.predict(X_tr_text), average='macro')
f1_cv = float(svc_search.best_score_)
f1_eyeball = f1_score(y_eye_sent, svc_sent_pipe.predict(X_eye_text), average='macro')
f1_blackbox = f1_score(y_blk_sent, svc_sent_pipe.predict(X_blk_text), average='macro')
print(f'SVC Sent -> Train F1: {f1_train:.4f} | CV F1: {f1_cv:.4f} | Eyeball F1: {f1_eyeball:.4f} | Blackbox F1: {f1_blackbox:.4f}')
svc_sent_scores = {'model': 'LinearSVC', 'pipe': svc_sent_pipe, 'train_f1': f1_train, 'cv_f1': f1_cv, 'eyeball_f1': f1_eyeball, 'blackbox_f1': f1_blackbox}

SVC Sent -> Train F1: 0.9267 | CV F1: 0.7145 | Eyeball F1: 0.7369 | Blackbox F1: 0.7469


In [24]:
#Tu dong chon mo hinh co Blackbox Dev Macro F1 cao nhat (khong gan cung)
sent_candidates = [nb_sent_scores, lr_sent_scores, svc_sent_scores]
df_sent_cmp = pd.DataFrame([{k: v for k, v in r.items() if k != 'pipe'} for r in sent_candidates])
best_sent_row = max(sent_candidates, key=lambda r: r['blackbox_f1'])
best_sent_name, best_sent_pipe = best_sent_row['model'], best_sent_row['pipe']
print('Winner Sentiment on Blackbox Dev:', best_sent_name, '| Blackbox F1:', round(best_sent_row['blackbox_f1'], 4))
df_sent_cmp

Winner Sentiment on Blackbox Dev: LogisticRegression | Blackbox F1: 0.7621


,model,train_f1,cv_f1,eyeball_f1,blackbox_f1
0,ComplementNB,0.888008,0.699719,0.708987,0.732450
1,LogisticRegression,0.911123,0.731013,0.736316,0.762148
2,LinearSVC,0.926724,0.714464,0.736938,0.746857


## MÔ HÌNH ĐA KHÍA CẠNH (MULTI-LABEL ONE-VS-REST)

In [25]:
from sklearn.multiclass import OneVsRestClassifier
#Dung 5-fold phan tang theo khia canh chinh tren df_train de tune Multi-Label OneVsRestClassifier
asp_cv = list(skf.split(X_tr_text, y_tr_asp))
len(asp_cv)

5

In [26]:
#1. OneVsRestClassifier(LogisticRegression) trong Pipeline + 5-fold CV tren df_train
pipe_ovr_lr = Pipeline([('tfidf', TfidfVectorizer(**tfidf_params)), ('clf', OneVsRestClassifier(LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)))])
ovr_lr_grid = {'clf__estimator__C': [0.5, 1.0, 2.0]}
ovr_lr_search = GridSearchCV(pipe_ovr_lr, ovr_lr_grid, cv=asp_cv, scoring='f1_macro', n_jobs=-1, refit=True)
ovr_lr_search.fit(X_tr_text, Y_tr_asp)
ovr_lr_pipe = ovr_lr_search.best_estimator_
print('OVR LR best params:', ovr_lr_search.best_params_)

OVR LR best params: {'clf__estimator__C': 2.0}


In [27]:
ml_tr = f1_score(Y_tr_asp, ovr_lr_pipe.predict(X_tr_text), average='macro')
ml_cv = float(ovr_lr_search.best_score_)
ml_eye = f1_score(Y_eye_asp, ovr_lr_pipe.predict(X_eye_text), average='macro')
ml_blk = f1_score(Y_blk_asp, ovr_lr_pipe.predict(X_blk_text), average='macro')
blk_prim_lr = multilabel_to_primary_aspect(ovr_lr_pipe.predict_proba(X_blk_text), threshold=0.5)
prim_blk_lr = f1_score(y_blk_asp, blk_prim_lr, average='macro')
hit_blk_lr = float(Y_blk_asp[np.arange(len(Y_blk_asp)), blk_prim_lr].mean())
print(f'OVR LR -> Train ML-F1: {ml_tr:.4f} | CV ML-F1: {ml_cv:.4f} | Eyeball ML-F1: {ml_eye:.4f} | Blackbox ML-F1: {ml_blk:.4f} | Prim-F1: {prim_blk_lr:.4f} | Hit-Any: {hit_blk_lr:.4f}')
ovr_lr_scores = {'model': 'OVR_LogisticRegression', 'pipe': ovr_lr_pipe, 'train_ml_f1': ml_tr, 'cv_ml_f1': ml_cv, 'eyeball_ml_f1': ml_eye, 'blackbox_ml_f1': ml_blk, 'blackbox_prim_f1': prim_blk_lr, 'blackbox_hit_any': hit_blk_lr}

OVR LR -> Train ML-F1: 0.9586 | CV ML-F1: 0.8904 | Eyeball ML-F1: 0.8815 | Blackbox ML-F1: 0.8949 | Prim-F1: 0.4887 | Hit-Any: 0.9500


In [28]:
#2. OneVsRestClassifier(LinearSVC) trong Pipeline + 5-fold CV tren df_train
pipe_ovr_svc = Pipeline([('tfidf', TfidfVectorizer(**tfidf_params)), ('clf', OneVsRestClassifier(LinearSVC(max_iter=3000, class_weight='balanced', random_state=42)))])
ovr_svc_grid = {'clf__estimator__C': [0.1, 0.3, 0.5, 1.0]}
ovr_svc_search = GridSearchCV(pipe_ovr_svc, ovr_svc_grid, cv=asp_cv, scoring='f1_macro', n_jobs=-1, refit=True)
ovr_svc_search.fit(X_tr_text, Y_tr_asp)
ovr_svc_pipe = ovr_svc_search.best_estimator_
print('OVR SVC best params:', ovr_svc_search.best_params_)

OVR SVC best params: {'clf__estimator__C': 1.0}


In [29]:
ml_tr = f1_score(Y_tr_asp, ovr_svc_pipe.predict(X_tr_text), average='macro')
ml_cv = float(ovr_svc_search.best_score_)
ml_eye = f1_score(Y_eye_asp, ovr_svc_pipe.predict(X_eye_text), average='macro')
ml_blk = f1_score(Y_blk_asp, ovr_svc_pipe.predict(X_blk_text), average='macro')
blk_prim_svc = multilabel_to_primary_aspect(expit(ovr_svc_pipe.decision_function(X_blk_text)), threshold=0.5)
prim_blk_svc = f1_score(y_blk_asp, blk_prim_svc, average='macro')
hit_blk_svc = float(Y_blk_asp[np.arange(len(Y_blk_asp)), blk_prim_svc].mean())
print(f'OVR SVC -> Train ML-F1: {ml_tr:.4f} | CV ML-F1: {ml_cv:.4f} | Eyeball ML-F1: {ml_eye:.4f} | Blackbox ML-F1: {ml_blk:.4f} | Prim-F1: {prim_blk_svc:.4f} | Hit-Any: {hit_blk_svc:.4f}')
ovr_svc_scores = {'model': 'OVR_LinearSVC', 'pipe': ovr_svc_pipe, 'train_ml_f1': ml_tr, 'cv_ml_f1': ml_cv, 'eyeball_ml_f1': ml_eye, 'blackbox_ml_f1': ml_blk, 'blackbox_prim_f1': prim_blk_svc, 'blackbox_hit_any': hit_blk_svc}

OVR SVC -> Train ML-F1: 0.9947 | CV ML-F1: 0.8967 | Eyeball ML-F1: 0.8925 | Blackbox ML-F1: 0.9021 | Prim-F1: 0.4730 | Hit-Any: 0.9554


In [30]:
#Tu dong chon mo hinh Multi-Label Aspect co Blackbox Dev Multi-label Macro F1 cao nhat
asp_candidates = [ovr_lr_scores, ovr_svc_scores]
df_asp_cmp = pd.DataFrame([{k: v for k, v in r.items() if k != 'pipe'} for r in asp_candidates])
best_asp_row = max(asp_candidates, key=lambda r: r['blackbox_ml_f1'])
best_asp_name, best_asp_pipe = best_asp_row['model'], best_asp_row['pipe']
print('Winner Aspect on Blackbox Dev:', best_asp_name, '| Blackbox ML-F1:', round(best_asp_row['blackbox_ml_f1'], 4))
df_asp_cmp

Winner Aspect on Blackbox Dev: OVR_LinearSVC | Blackbox ML-F1: 0.9021


,model,train_ml_f1,cv_ml_f1,eyeball_ml_f1,blackbox_ml_f1,blackbox_prim_f1,blackbox_hit_any
0,OVR_LogisticRegression,0.958585,0.890413,0.881451,0.894926,0.488664,0.950000
1,OVR_LinearSVC,0.994686,0.896681,0.892511,0.902064,0.472989,0.955357


## PHÂN TÍCH LỖI TRÊN TẬP EYEBALL DEV (ANDREW NG)

In [31]:
#Chi soi loi tren df_eyeball (250 mau), tuyet doi khong nhin df_blackbox (560 mau)
pred_eye_sent = best_sent_pipe.predict(X_eye_text)
df_err = df_eyeball[y_eye_sent != pred_eye_sent].copy()
df_err['pred_sent'] = pred_eye_sent[y_eye_sent != pred_eye_sent]
print('So cau doan sai tren Eyeball Dev:', len(df_err), '/', len(df_eyeball))

So cau doan sai tren Eyeball Dev:

 55 / 250


In [32]:
df_err[['n_star', 'sentiment', 'pred_sent', 'sentence_clean']].head(10)

,n_star,sentiment,pred_sent,sentence_clean
898,1,1,2,"Nhân viên cửa hàng 446 Đại Mỗ, Nam Từ Liêm, Hà..."
936,2,1,0,Mua máy 15/4 mà sao chơi game thoát ra là bị đ...
287,4,0,1,Cằm dày rất nhiều so với a10. Còn về hiệu năng...
181,1,1,0,Mau hết pin quá..4500ma.mà mau hết quá tr..45%...
362,1,0,1,Hình như 16 củ thì không đáng đồng tiền bát gạ...
757,5,2,1,Điện thoại sài ngon lắm. Chơi liên quân good. ...
939,2,1,0,"Máy bị lỗi sim , đổi máy rồi vẫn bị, chụp hình..."
411,4,0,1,"Vừa mua realme c3i được hơn 1 tuần , chip medi..."
585,4,2,1,Mình mới mua máy hôm 15/02. Trải nghiệm 2 ngày...
826,5,1,0,E mua con này về dùng ngon nhưng khi nâng cấp ...


In [33]:
#Thu bieu thuc chinh quy bat tu noi doi lap tren 1 mau loi
tempt = df_err['sentence_clean'].iloc[0]
bool(re.search(r'(?<!\w)(?:nhưng|tuy nhiên|có điều|chỉ có|mặc dù|dù|trừ|còn lại)(?!\w)', str(tempt).lower()))

False

In [34]:
def classify_err(t):
    s = str(t).lower()
    if re.search(r'(?<!\w)(?:nhưng|tuy nhiên|có điều|chỉ có|mặc dù|dù|trừ|còn lại)(?!\w)', s): return '1. Câu đa ý trái chiều / chuyển ý (nhưng, tuy nhiên, chỉ có điều)'
    if re.search(r'(?<!\w)(?:không|chưa|chẳng|chả|đâu có|kém|hơi)(?!\w)', s): return '2. Phủ định / đảo ngữ phức tạp'
    if len(s.split()) < 15: return '3. Câu ngắn / hàm ý thiếu từ khóa'
    return '4. Ẩn dụ / Từ vựng hiếm / Khác'
df_err['error_type'] = df_err['sentence_clean'].apply(classify_err)

In [35]:
err_cnt = df_err['error_type'].value_counts()
err_pct = (err_cnt / max(len(df_err), 1) * 100).round(1)
pd.DataFrame({'so_luong_loi': err_cnt, 'ty_le_phan_tram': err_pct})

,so_luong_loi,ty_le_phan_tram
error_type,,
2. Phủ định / đảo ngữ phức tạp,21,38.2
"1. Câu đa ý trái chiều / chuyển ý (nhưng, tuy nhiên, chỉ có điều)",19,34.5
4. Ẩn dụ / Từ vựng hiếm / Khác,14,25.5
3. Câu ngắn / hàm ý thiếu từ khóa,1,1.8


## ĐÁNH GIÁ NIÊM PHONG TRÊN TẬP TEST VÀ LƯU MÔ HÌNH

In [36]:
test_pred_sent = best_sent_pipe.predict(X_te_text)
print('Test Sentiment Macro F1:', round(f1_score(y_te_sent, test_pred_sent, average='macro'), 4))
print('Test Sentiment Accuracy:', round(accuracy_score(y_te_sent, test_pred_sent), 4))
print(classification_report(y_te_sent, test_pred_sent, target_names=['Tiêu cực', 'Trung lập', 'Tích cực'], digits=4))

Test Sentiment Macro F1: 0.7345
Test Sentiment Accuracy: 0.7824
              precision    recall  f1-score   support

    Tiêu cực     0.8012    0.8690    0.8338       756
   Trung lập     0.4978    0.5126    0.5051       435
    Tích cực     0.8996    0.8325    0.8648      1033

    accuracy                         0.7824      2224
   macro avg     0.7329    0.7381    0.7345      2224
weighted avg     0.7876    0.7824    0.7839      2224



In [37]:
#Suy dien Multi-Label Aspect va quy doi Primary Aspect voi fallback argmax khi < threshold
Y_te_pred_asp = best_asp_pipe.predict(X_te_text)
test_asp_scores = best_asp_pipe.predict_proba(X_te_text) if hasattr(best_asp_pipe, 'predict_proba') else expit(best_asp_pipe.decision_function(X_te_text))
test_pred_asp = multilabel_to_primary_aspect(test_asp_scores, threshold=0.5)
test_ml_f1 = f1_score(Y_te_asp, Y_te_pred_asp, average='macro')
test_prim_f1 = f1_score(y_te_asp, test_pred_asp, average='macro')
test_hit_any = float(Y_te_asp[np.arange(len(Y_te_asp)), test_pred_asp].mean())
print(f'Test Multi-label Macro F1: {test_ml_f1:.4f} | Primary Aspect Macro F1: {test_prim_f1:.4f} | Hit-Any-True-Group: {test_hit_any:.4f}')

Test Multi-label Macro F1: 0.8925 | Primary Aspect Macro F1: 0.5100 | Hit-Any-True-Group: 0.9577


In [38]:
print('--- MULTI-LABEL ASPECT REPORT ---')
print(classification_report(Y_te_asp, Y_te_pred_asp, target_names=ASPECT_GROUPS, digits=4, zero_division=0))
print('--- PRIMARY ASPECT REPORT ---')
print(classification_report(y_te_asp, test_pred_asp, target_names=ASPECT_GROUPS, digits=4))

--- MULTI-LABEL ASPECT REPORT ---
              precision    recall  f1-score   support

   PERF_FEAT     0.9201    0.8951    0.9074      1544
  SCREEN_CAM     0.9127    0.8920    0.9022       750
     BATTERY     0.9568    0.9389    0.9477      1014
   PRICE_SER     0.8877    0.8228    0.8540       999
  DESIGN_GEN     0.8805    0.8236    0.8511      1780

   micro avg     0.9090    0.8692    0.8886      6087
   macro avg     0.9115    0.8745    0.8925      6087
weighted avg     0.9084    0.8692    0.8883      6087
 samples avg     0.9012    0.8631    0.8627      6087

--- PRIMARY ASPECT REPORT ---
              precision    recall  f1-score   support

   PERF_FEAT     0.7112    0.6094    0.6564       873
  SCREEN_CAM     0.5088    0.4113    0.4549       282
     BATTERY     0.6432    0.5182    0.5740       494
   PRICE_SER     0.4885    0.5247    0.5060       324
  DESIGN_GEN     0.2689    0.5378    0.3586       251

    accuracy                         0.5436      2224
   macro avg 

In [39]:
#Fit cac mo hinh doc lap tren X_tr_tfidf de tuong thich SHAP (NB05) va Excel (NB06)
lr_best_kw = {k.replace('clf__', ''): v for k, v in lr_search.best_params_.items()}
svc_best_kw = {k.replace('clf__', ''): v for k, v in svc_search.best_params_.items()}
lr_sent = LogisticRegression(**lr_best_kw, max_iter=1000, random_state=42).fit(X_tr_tfidf, y_tr_sent)
svc_sent = LinearSVC(**svc_best_kw, max_iter=3000, random_state=42).fit(X_tr_tfidf, y_tr_sent)
svc_asp = LinearSVC(C=ovr_svc_search.best_params_['clf__estimator__C'], class_weight='balanced', max_iter=3000, random_state=42).fit(X_tr_tfidf, y_tr_asp)

In [40]:
#Luu pipeline thang giai va cac mo hinh tuong thich ra models/
joblib.dump(best_sent_pipe, os.path.join(MODEL_DIR, 'baseline_sentiment.pkl'))
joblib.dump(best_asp_pipe, os.path.join(MODEL_DIR, 'baseline_aspect_multilabel.pkl'))
joblib.dump(vec, os.path.join(MODEL_DIR, 'tfidf_vectorizer.pkl'))
joblib.dump(lr_sent, os.path.join(MODEL_DIR, 'lr_sentiment.pkl'))
joblib.dump(svc_sent, os.path.join(MODEL_DIR, 'svc_sentiment.pkl'))
joblib.dump(svc_asp, os.path.join(MODEL_DIR, 'svc_aspect.pkl'))

['D:\\E-commerce\\models\\svc_aspect.pkl']

In [41]:
df_baseline_pred = df_test.copy()
df_baseline_pred['pred_sentiment'] = test_pred_sent
for i in range(5):
    df_baseline_pred[f'pred_asp_{i}'] = Y_te_pred_asp[:, i]
df_baseline_pred['pred_aspect'] = test_pred_asp
out_csv = os.path.join(PROC_DIR, 'test_pred_baseline.csv')
df_baseline_pred.to_csv(out_csv, index=False, encoding='utf-8-sig')
df_baseline_pred.shape

(2224, 23)

In [42]:
sorted(os.listdir(MODEL_DIR))

['baseline_aspect_multilabel.pkl',
 'baseline_sentiment.pkl',
 'lr_aspect.pkl',
 'lr_sentiment.pkl',
 'model_metadata.json',
 'phobert_multitask',
 'svc_aspect.pkl',
 'svc_sentiment.pkl',
 'tfidf_vectorizer.pkl']

In [43]:
pd.read_csv(out_csv)[['sentence_clean', 'sentiment', 'pred_sentiment', 'aspect_group', 'pred_aspect', 'pred_asp_0', 'pred_asp_1', 'pred_asp_2', 'pred_asp_3', 'pred_asp_4']].head(3)

,sentence_clean,sentiment,pred_sentiment,aspect_group,pred_aspect,pred_asp_0,pred_asp_1,pred_asp_2,pred_asp_3,pred_asp_4
0,"Điện thoải ổn. Facelock cực nhanh, vân tay ôk ...",2,2,0,0,1,1,1,0,1
1,"Mình mới mua vivo91c. Tải ứng dụng ,games nhan...",2,1,0,4,1,0,0,0,1
2,Xấu đẹp gì không biết nhưng rất ưng Thế Giới D...,1,2,4,3,0,0,0,1,1
